# Chỉnh sửa đồ thị cảnh theo một yêu cầu

Chạy **Restart Kernel and Run All Cells** bằng môi trường `myenv` của dự án. Notebook dùng checkpoint, trọng số CLIP, cache văn bản MagicBrush và dữ liệu VN100 có sẵn trong workspace. Nếu thiếu manifest VN100, chạy `bash scripts/package_level3.sh prepare` từ thư mục gốc.

Một ảnh và một câu tiếng Anh được dùng xuyên suốt. GNN huấn luyện trên MagicBrush dự đoán thao tác trên graph nguồn VN100 ngay trong notebook. Đây là minh họa chuyển miền trên một ca. Ảnh chỉnh sửa do Qwen tạo độc lập từ ảnh nguồn và câu yêu cầu. Các graph REACT++ và ảnh Qwen được đọc từ lần chạy đã lưu.

In [ ]:
import json
import os
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = ""
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "tools/graph_editing_live_demo_v1.py").is_file())
sys.path.insert(0, str(ROOT))

In [ ]:
import torch
from IPython.display import HTML, Image, display
from src.editing.operations import EditGraph
from tools.graph_editing_live_demo_v1 import predict
from tools.level3_submission_v1.demo import checked, graph_html, image_pair_html, open_bundle, recompute, table

torch.set_num_threads(2)
root, manifest = open_bundle(ROOT)

## 1. Đọc ảnh, yêu cầu và đồ thị nguồn

Ca mặc định yêu cầu thêm thuyền buồm ở tiền cảnh. Để thử giới hạn của bộ chấm màu, đổi `TASK_ID` thành `sidewalk-b1bb633496439f-q200-2` rồi chạy lại từ ô dưới. Trong bảng cạnh, đọc mỗi hàng từ chủ thể qua quan hệ đến đối tượng.

In [ ]:
TASK_ID = "travel-7bb0c6b3635205-q100-1"
case = next(c for c in manifest["cases"] if c["task_id"] == TASK_ID)
print(case["task_id"], case["instruction"], sep="\n")
display(Image(filename=str(checked(root, manifest, case["source"])), width=520))
source_record = json.loads(checked(root, manifest, case["graphs"]["source"]).read_text())
source = EditGraph.from_dict(source_record["source_graph"])
display(HTML(graph_html(source.to_dict(), "Đồ thị nguồn do REACT++ dự đoán", "source-arrow")))

## 2. Dự đoán thao tác và dựng đồ thị đích

GNN nhận đồ thị nguồn cùng câu yêu cầu. Bộ giải mã chọn thao tác rồi áp dụng chúng lên bản sao của đồ thị nguồn. Đối chiếu nhãn và thuộc tính của đối tượng được sửa với danh từ, thuộc tính và vị trí trong câu yêu cầu.

In [ ]:
operations, predicted, selection = predict(source, case["instruction"])
print(f"GNN MagicBrush · prompt · seed {selection['seed']} · epoch {selection['selected_epoch']}")
print("Checkpoint SHA-256:", selection["sha256"])
print("Thao tác dự đoán:", json.dumps([op.to_dict() for op in operations], ensure_ascii=False, indent=2))
display(HTML(graph_html(predicted.to_dict(), "Đồ thị đích dựng từ thao tác GNN", "gnn-arrow")))

## 3. Quan sát ảnh sau chỉnh sửa và tính điểm nhất quán

Khung đỏ biểu diễn đối tượng REACT++ phát hiện. Khung vàng trên ảnh Qwen đánh dấu vùng chỉnh sửa dự kiến. REACT++ trích đồ thị riêng cho từng ảnh, nên cùng một ID có thể chỉ hai đối tượng khác nhau. Bộ chấm ghép đối tượng theo nhãn và độ giao nhau của hộp, kiểm tra điều kiện chỉnh sửa (E) và phần đồ thị được bảo toàn (P). Khi đủ bằng chứng, G = E × P. Điểm này đánh giá **cặp ảnh nguồn–Qwen**.

In [ ]:
scored = recompute(root, manifest, TASK_ID)
display(HTML(image_pair_html(root, scored)))
display(HTML(graph_html(scored["graphs_loaded"]["target"], "Đồ thị trích từ ảnh Qwen", "qwen-arrow")))
s, labels = scored["recomputed"], scored["scores_and_labels"]

def show_score(value):
    return "Không chấm được" if value is None else f"{value:.3f}"

display(HTML(table(["E", "P", "G", "Nhãn tay: đúng yêu cầu", "Nhãn tay: bảo toàn"],
                   [[show_score(s["edit_score"]), show_score(s["preservation_score"]),
                     show_score(s["graph_consistency"]), labels["edit_fulfillment"], labels["preservation"]]])))
print("Điều kiện chỉnh sửa:", s.get("checks", []))
print("P đủ bằng chứng:", s["preservation_evaluable"])
print("Lý do hạn chế:", s["reason_codes"] or "—")

**Đọc kết quả ca thuyền buồm.** Graph nguồn đã có một nút `boat`. Hãy xem nút mới trong đồ thị GNN: checkpoint dự đoán thêm `giraffe`, sai đối tượng yêu cầu. Ở nhánh ảnh, Qwen nhận trực tiếp câu yêu cầu thêm thuyền. Bộ chấm cho cặp ảnh G = 1 và người duyệt đánh dấu đạt cả chỉnh sửa lẫn bảo toàn. Một ca như vậy minh họa cách đọc đầu ra từng mô hình; đánh giá định lượng trên tập kiểm tra nằm trong báo cáo.

**Đọc ca đổi màu ghế.** REACT++ không cung cấp thuộc tính màu. Bộ chấm trả về E và G không xác định, dù vẫn có thể đo phần đồ thị được bảo toàn. Người duyệt có thể kiểm tra màu trực tiếp trên ảnh.